In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
# Multi-layer structural path verification to completely solve blank execution loops
possible_paths = [
    'netflix_titles.csv',
    os.path.join('data', 'netflix_titles.csv'),
    os.path.join('..', 'data', 'netflix_titles.csv')
]

data_path = None
for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError("CRITICAL ERROR: Could not locate 'netflix_titles.csv' in any portfolio directory layout.")

print(f"Success! Loading target stream matrix from: {data_path}")
netflix_df = pd.read_csv(data_path)
netflix_df.head()

In [3]:
netflix_df.columns

### Tier 1: Data Hygiene & Catalog Diagnostics

In [4]:
netflix_df.info()

In [5]:
# Missing metrics evaluation
round(netflix_df.isna().sum()/len(netflix_df),2)

In [6]:
# Structural clean-up
netflix_df.drop(['show_id','description'], axis=1, inplace=True, errors='ignore')

In [7]:
# Dispersion Diagnostics
rng = netflix_df['release_year'].max() - netflix_df['release_year'].min()
var = netflix_df['release_year'].var()
std = netflix_df['release_year'].std()
print(f'Range of the release year: {rng} years')
print(f'Variance: {var}')
print(f'Standard Deviation: {std}')

In [8]:
# Visual catalog profile density distribution
plt.figure(figsize=(10, 5))
sns.histplot(data=netflix_df, x='release_year', bins=30, kde=True)
plt.title('Netflix Catalog Release Distribution (Historical Dispersion Overview)')
plt.xlabel('Release Year')
plt.ylabel('Title Count')
plt.show()

### Tier 2: Audience Segmentation & Format Cross-Tabulation

In [9]:
# Imputing rating sparsity with statistical mode validation
mode_rating = netflix_df['rating'].mode()[0]
netflix_df['rating'].fillna(value=mode_rating, inplace=True)

In [10]:
# Multi-Variable Cross-Tabulation
audience_matrix = pd.crosstab(netflix_df['type'], netflix_df['rating'], normalize='index')
print("--- Row-Normalized Conditional Probabilities (Strategic View) ---")
print(round(audience_matrix * 100, 2))

In [11]:
plt.figure(figsize=(14, 6))
sns.heatmap(round(audience_matrix*100, 2), annot=True, cmap='viridis', fmt='.2f')
plt.title('Conditional Audience Profile Density Split: Format vs Maturity Standard Matrix')
plt.show()

### Tier 3: Growth Ingestion Velocity Timelines

In [12]:
netflix_df['date_added'] = pd.to_datetime(netflix_df['date_added'].str.strip())
netflix_df['year_added'] = netflix_df['date_added'].dt.year
netflix_df['month_added'] = netflix_df['date_added'].dt.month

In [13]:
plt.figure(figsize=(10, 5))
titles_per_year = netflix_df.groupby('year_added').size()
titles_per_year.plot(kind='bar', color='skyblue')
plt.title('Global Inventory Ingestion Scale Vector over Time')
plt.xlabel('Calendar Year Added')
plt.ylabel('Abs Vol of Content Onboarded')
plt.show()

### Tier 4: Genre Explosion & Thematic Anchors

In [14]:
# Multi-label categorical string tokenization
netflix_df['listed_in'] = netflix_df['listed_in'].str.split(', ')
genre_df = netflix_df.explode('listed_in')

plt.figure(figsize=(14, 6))
genre_df['listed_in'].value_counts().plot(kind='bar', color='salmon')
plt.title('Strategic Corporate Core Thematic Anchors (Exploded Value Breakdown)')
plt.ylabel('Aggregated Total Volume Occurrence')
plt.show()

### Tier 5: Duration Parsing & Content Longevity Funnel Decay

In [15]:
# Feature subset generation and clean casting
movies_df = netflix_df[netflix_df['type'] == 'Movie'].copy()
movies_df['dur'] = movies_df['duration'].str.replace(' min', '', case=False).fillna('0')
movies_df['dur'] = pd.to_numeric(movies_df['dur'], errors='coerce').fillna(0).astype(int)

tv_df = netflix_df[netflix_df['type'] == 'TV Show'].copy()
tv_df['dur'] = tv_df['duration'].str.extract(r'(\d+)').fillna('0')
tv_df['dur'] = tv_df['dur'].astype(int)

In [16]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(data=movies_df, x='dur', bins=20, ax=axes[0], kde=True, color='purple')
axes[0].set_title('Feature Film Continuous Runtime Distribution Summary')
axes[0].set_xlabel('Duration (Minutes)')

tv_decay = tv_df['dur'].value_counts(normalize=True).sort_index()
tv_decay.plot(kind='bar', ax=axes[1], color='teal')
axes[1].set_title('Episodic Content Lifecycle Drop-off Matrix (Retention Funnel Decay)')
axes[1].set_xlabel('Lifespan Cap (Total Completed Seasons)')
axes[1].set_ylabel('Proportional Share of Catalog')

plt.tight_layout()
plt.show()